# Day 2: Streaming Ingestion (Kafka / MSK + Spark Streaming)

This notebook simulates streaming ingestion by generating synthetic FHIR data and appending to the Bronze Iceberg table via PySpark memory streams.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, lit, expr
import os
import time

warehouse_path = os.path.join(os.getcwd(), "spark-warehouse")

spark = SparkSession.builder \
    .appName("Day02_Streaming_Ingestion") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.local.type", "hadoop") \
    .config("spark.sql.catalog.local.warehouse", warehouse_path) \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("Spark session created.")

In [ ]:
# Mock memory stream as rate source
rate_df = spark.readStream \
    .format("rate") \
    .option("rowsPerSecond", 5) \
    .load()

transformed_df = rate_df \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("source_system_id", lit("memory_stream_simulator")) \
    .withColumn("payload_id", expr("uuid()")) \
    .withColumn("raw_payload_json", expr("concat('{\"resourceType\": \"Patient\", \"id\": \"', uuid(), '\", \"active\": true}')")) \
    .select("ingestion_timestamp", "source_system_id", "payload_id", "raw_payload_json")

query = transformed_df.writeStream \
    .format("iceberg") \
    .outputMode("append") \
    .option("checkpointLocation", os.path.join(os.getcwd(), "checkpoints/fhir_stream")) \
    .trigger(processingTime="2 seconds") \
    .toTable("local.ehdip_bronze.raw_payloads")

print("Started streaming query...")
time.sleep(10)
query.stop()
print("Stopped streaming query.")

In [ ]:
spark.sql("SELECT * FROM local.ehdip_bronze.raw_payloads ORDER BY ingestion_timestamp DESC LIMIT 5").show(truncate=False)